In [1]:
import pandas as pd

In [2]:
df1 = pd.read_csv("raw_data/daily_20241108T0927_000.csv", skiprows=1, parse_dates=['Date'], date_format="%Y/%m/%d") 
# keep only the rows with valid value for column "Value"
df1 = df1[df1['Value'].notnull()]
# keep only the date part of our datetime object
df1['Date'] = df1['Date'].dt.date

print(type(df1['Date'][0]))
print("Start date: ", df1['Date'].min())
print("End date: ", df1['Date'].max())


<class 'datetime.date'>
Start date:  1911-01-28
End date:  2023-12-31


In [3]:
df1

,ID,PARAM,Date,Value,SYM
0,08LD001,1,1911-01-28,3.820,NaN
154,08LD001,1,1911-07-01,159.000,NaN
155,08LD001,1,1911-07-02,159.000,NaN
156,08LD001,1,1911-07-03,167.000,NaN
157,08LD001,1,1911-07-04,176.000,NaN
...,...,...,...,...,...
615613,08LB038,2,2023-12-27,0.804,NaN
615614,08LB038,2,2023-12-28,0.729,NaN
615615,08LB038,2,2023-12-29,0.710,NaN
615616,08LB038,2,2023-12-30,0.706,NaN


In [4]:
df2 = pd.read_csv("raw_data/daily_20241108T0928_001.csv", skiprows=1, parse_dates=['Date'], date_format="%Y/%m/%d") 
# keep only the rows with valid value for column "Value"
df2 = df2[df2['Value'].notnull()]
# keep only the date part of our datetime object
df2['Date'] = df2['Date'].dt.date

print(type(df2['Date'][0]))
print("Start date: ", df2['Date'].min())
print("End date: ", df2['Date'].max())


<class 'datetime.date'>
Start date:  1911-11-01
End date:  2023-12-31


In [5]:
print(len(df1))
print(len(df2))
print(len(joined_df))
joined_df.to_csv("test.csv")

514188
437088


NameError: name 'joined_df' is not defined

In [6]:
# split our data into two sets, one with PARAM == 1 and another with PARAM == 2
df1_1 = df1[df1['PARAM'] == 1]
df1_2 = df1[df1['PARAM'] == 2]
df2_1 = df2[df2['PARAM'] == 1]
df2_2 = df2[df2['PARAM'] == 2]
joined = df1_1.merge(df2_1, on="Date")
print(len(df1_1), len(df2_1), len(joined))
print(f'{len(df1_1)} + {len(df2_1)} = {len(df1_1) + len(df2_1)}')
print("Start date: ", df1_1['Date'].min())
print("End date: ", df1_1['Date'].max())
print("Start date: ", df2_1['Date'].min())
print("End date: ", df2_1['Date'].max())
print("Start date: ", joined['Date'].min())
print("End date: ", joined['Date'].max())

422772 354988 5343479
422772 + 354988 = 777760
Start date:  1911-01-28
End date:  2023-12-31
Start date:  1911-11-01
End date:  2023-12-31
Start date:  1911-11-01
End date:  2023-12-31


In [ ]:
df1_1.nunique()

 ID         20
PARAM        1
Date     39966
Value     3397
SYM          3
dtype: int64

In [ ]:
df2_1.nunique()

 ID         20
PARAM        1
Date     40903
Value     3540
SYM          3
dtype: int64

In [ ]:
joined.nunique()

 ID_x         20
PARAM_x        1
Date       39854
Value_x     3397
SYM_x          3
 ID_y         20
PARAM_y        1
Value_y     3540
SYM_y          3
dtype: int64

In [ ]:
joined.to_csv("test.csv")

### What do I want my df to look like after I'm finished cleaning it up?
| Date | StationID | Value | SYM |

However, I want Dates where I have data available for all Stations.
Can we just do a join on all of our dataframes and up with the intersection of all of our sets of date values?

In [ ]:
# try doing a pivot, to see if we can get dates as the index, and each stationID is a column, their intersection is the value.

df1_1

,ID,PARAM,Date,Value,SYM
0,08LD001,1,1911-01-28,3.82,NaN
154,08LD001,1,1911-07-01,159.00,NaN
155,08LD001,1,1911-07-02,159.00,NaN
156,08LD001,1,1911-07-03,167.00,NaN
157,08LD001,1,1911-07-04,176.00,NaN
...,...,...,...,...,...
610865,08LB038,1,2023-12-27,2.70,B
610866,08LB038,1,2023-12-28,2.26,B
610867,08LB038,1,2023-12-29,2.08,NaN
610868,08LB038,1,2023-12-30,2.02,NaN


In [14]:

# df1 = pd.read_csv("raw_data/daily_20241108T0927_000.csv", skiprows=1, parse_dates=['Date'], date_format="%Y/%m/%d") 
df1 = pd.read_csv("raw_data/daily_20241108T0927_000.csv", skiprows=1) 
# keep only the rows with valid value for column "Value"
df1 = df1[df1['Value'].notnull()]
# keep only the date part of our datetime object
# df1['Date'] = df1['Date'].dt.date
df1_1 = df1[df1['PARAM'] == 1]

pivot = df1_1.pivot(columns=" ID", index="Date")
# print(type(pivot.index[0]))
# datetime = pd.to_datetime('1911-01-28', format='%Y-%m-%d').date()
# print(type(datetime))
# print(pivot.loc['2023/12/31'])
# remove any columns that are completely empty, this is can happen since we filtered for datasets with "Flow and Level" data
# However, we then split our datasets into one set with the 'Flow' values, and another with the 'Level' values.
# Unfortunately, these values aren't always available at the same time for a given station. So when we do the pivot and set the
# index to be all possible dates in our dataset, we have a column(s) of all null
# pivot.dropna(how='all', axis=1, inplace=True)
# now lets drop all the remaining rows that still have some NaN values (partially filled columns
# keep rows that have at least 19 columns without NaN as a value
# pivot.dropna(axis=0, thresh=19,inplace=True)
pivot

PARAM                                                          \
 ID        07FC001 08DB013 08EC001 08EC013 08FB006 08FB011 08HA016 08KE016   
Date                                                                         
1911/01/28     NaN     NaN     NaN     NaN     NaN     NaN     NaN     NaN   
1911/07/01     NaN     NaN     NaN     NaN     NaN     NaN     NaN     NaN   
1911/07/02     NaN     NaN     NaN     NaN     NaN     NaN     NaN     NaN   
1911/07/03     NaN     NaN     NaN     NaN     NaN     NaN     NaN     NaN   
1911/07/04     NaN     NaN     NaN     NaN     NaN     NaN     NaN     NaN   
...            ...     ...     ...     ...     ...     ...     ...     ...   
2023/12/27     1.0     1.0     NaN     1.0     1.0     1.0     1.0     1.0   
2023/12/28     1.0     1.0     NaN     1.0     1.0     1.0     1.0     1.0   
2023/12/29     1.0     1.0     NaN     1.0     1.0     1.0     1.0     1.0   
2023/12/30     1.0     1.0     NaN     1.0     1.0     1.0     1.0     1.0   
2023/12/31     1.0     1.0     NaN     1.0     1.0     1.0     1.0     1.0   

                            ...     SYM                                  \
 ID        08LB020 08LB038  ... 08LB069 08LD001 08MH005 08NB012 08NE008   
Date                        ...                                           
1911/01/28     NaN     NaN  ...     NaN     NaN     NaN     NaN     NaN   
1911/07/01     NaN     NaN  ...     NaN     NaN     NaN     NaN     NaN   
1911/07/02     NaN     NaN  ...     NaN     NaN     NaN     NaN     NaN   
1911/07/03     NaN     NaN  ...     NaN     NaN     NaN     NaN     NaN   
1911/07/04     NaN     NaN  ...     NaN     NaN     NaN     NaN     NaN   
...            ...     ...  ...     ...     ...     ...     ...     ...   
2023/12/27     1.0     1.0  ...     NaN     NaN     NaN       B     NaN   
2023/12/28     1.0     1.0  ...     NaN     NaN     NaN       B     NaN   
2023/12/29     1.0     1.0  ...     NaN     NaN     NaN       B     NaN   
2023/12/30     1.0     1.0  ...     NaN     NaN     NaN       B     NaN   
2023/12/31     1.0     1.0  ...     NaN     NaN     NaN       B     NaN   

                                                    
 ID        08NH084 08NJ130 08NL004 09AA006 10CD005  
Date                                                
1911/01/28     NaN     NaN     NaN     NaN     NaN  
1911/07/01     NaN     NaN     NaN     NaN     NaN  
1911/07/02     NaN     NaN     NaN     NaN     NaN  
1911/07/03     NaN     NaN     NaN     NaN     NaN  
1911/07/04     NaN     NaN     NaN     NaN     NaN  
...            ...     ...     ...     ...     ...  
2023/12/27     NaN     NaN     NaN     NaN       B  
2023/12/28     NaN     NaN     NaN     NaN       B  
2023/12/29     NaN     NaN     NaN     NaN       B  
2023/12/30     NaN     NaN     NaN     NaN       B  
2023/12/31     NaN     NaN     NaN     NaN       B  

[39966 rows x 60 columns]

In [15]:
df1_1

,ID,PARAM,Date,Value,SYM
0,08LD001,1,1911/01/28,3.82,NaN
154,08LD001,1,1911/07/01,159.00,NaN
155,08LD001,1,1911/07/02,159.00,NaN
156,08LD001,1,1911/07/03,167.00,NaN
157,08LD001,1,1911/07/04,176.00,NaN
...,...,...,...,...,...
610865,08LB038,1,2023/12/27,2.70,B
610866,08LB038,1,2023/12/28,2.26,B
610867,08LB038,1,2023/12/29,2.08,NaN
610868,08LB038,1,2023/12/30,2.02,NaN


In [8]:
# want my index to be the date
# multi header
#    Date     |----------------------------------|Station ID---------------------------------|
#  2023-12-29 |longitude|latitude|Flow|Height|Precipitation|Temperature|Min/Max of those|etc.|
index = pd.MultiIndex.from_frame(pivot)
print(index)

MultiIndex([(  4.8,  1.44, nan, 46.8, 30.9, 76.2, 0.026,  1.48,  3.3, ...),
            ( 1.78,  0.28, nan, 34.9, 33.6, 88.5, 0.791, 0.837, 3.14, ...),
            (0.949, 0.373, nan, 27.8, 19.2, 44.9,  1.22,  1.08, 2.82, ...),
            (0.698, 0.203, nan, 22.9, 12.3, 28.8, 0.531,  1.85, 2.42, ...),
            ( 0.81, 0.285, nan, 23.1, 12.6, 31.0, 0.958,  1.99, 2.46, ...),
            (0.939, 0.342, nan, 23.4, 12.8, 31.7, 0.894,  2.14, 2.53, ...),
            ( 1.09, 0.297, nan, 23.2, 12.9, 31.3, 0.746,  2.29, 2.63, ...),
            ( 1.26, 0.269, nan, 22.9, 13.0, 30.7, 0.876,  2.28, 2.76, ...),
            ( 1.47, 0.261, nan, 23.1, 13.4, 30.8, 0.712,  2.28, 2.87, ...),
            (  1.7, 0.273, nan, 22.8, 14.0, 33.3, 0.661,  2.29, 2.94, ...),
            ...
            (0.743, 0.376, nan, 10.0, 6.62, 31.6, 0.497,  1.01, 3.07, ...),
            (0.732, 0.312, nan, 9.66, 6.42, 31.6, 0.428,  1.03, 3.02, ...),
            (0.721, 0.301, nan, 9.68, 6.47, 29.2, 0.392,  1.11, 2.93, ..

In [25]:
import csv
data = []
with open("raw_data/daily_20241108T0927_000.csv", newline='') as f:
    reader = csv.reader(f)
    csvFile = list(reader)
    data.append(csvFile)

data

[[['\ufeff\ufeffDaily Discharge (m3/s) (PARAM = 1) and Daily Water Level (m) (PARAM = 2)'],
  [' ID', 'PARAM', 'Date', 'Value', 'SYM'],
  ['08LD001', '1', '1911/01/28', '3.82', ''],
  ['08LD001', '1', '1911/01/29', '', ''],
  ['08LD001', '1', '1911/01/30', '', ''],
  ['08LD001', '1', '1911/01/31', '', ''],
  ['08LD001', '1', '1911/02/01', '', ''],
  ['08LD001', '1', '1911/02/02', '', ''],
  ['08LD001', '1', '1911/02/03', '', ''],
  ['08LD001', '1', '1911/02/04', '', ''],
  ['08LD001', '1', '1911/02/05', '', ''],
  ['08LD001', '1', '1911/02/06', '', ''],
  ['08LD001', '1', '1911/02/07', '', ''],
  ['08LD001', '1', '1911/02/08', '', ''],
  ['08LD001', '1', '1911/02/09', '', ''],
  ['08LD001', '1', '1911/02/10', '', ''],
  ['08LD001', '1', '1911/02/11', '', ''],
  ['08LD001', '1', '1911/02/12', '', ''],
  ['08LD001', '1', '1911/02/13', '', ''],
  ['08LD001', '1', '1911/02/14', '', ''],
  ['08LD001', '1', '1911/02/15', '', ''],
  ['08LD001', '1', '1911/02/16', '', ''],
  ['08LD001', '1', '

In [32]:
import os
import sys
raw_data_path = "raw_data"
clean_data_path = "clean_data"
if not os.path.isdir(clean_data_path):
    os.makedirs(clean_data_path)
# data = pd.DataFrame()
# # TODO specify a schema so that we dont get a DtypeWawrning of columns having mixed types
# for filename in os.listdir(raw_data_path):
#     print(filename)
#     df = pd.read_csv(f'raw_data/{filename}', skiprows=1)
#     df = df[df['Value'].notnull()]
#     print(f'Input file {filename} has {len(df)} non-null elements.')
#     # df_new.to_csv(f'{clean_data_path}/{filename}')

data = []
for filename in os.listdir(raw_data_path):
    with open(f'{raw_data_path}\\{filename}', newline='') as f:
        reader = list(csv.reader(f))
        print(f'Input file {filename} has {len(list(reader))} non-null elements.')
        data.append(reader)
            
        
print(len(data[0]))

Input file daily_20241108T0927_000.csv has 615620 non-null elements.
Input file daily_20241108T0928_001.csv has 570158 non-null elements.
Input file daily_20241108T0931_002.csv has 648480 non-null elements.
Input file daily_20241108T0932_003.csv has 561924 non-null elements.
Input file daily_20241108T0933_004.csv has 608006 non-null elements.
Input file daily_20241108T0933_005.csv has 482439 non-null elements.
Input file daily_20241108T0934_006.csv has 523782 non-null elements.
Input file daily_20241108T0935_007.csv has 512855 non-null elements.
Input file daily_20241108T0936_008.csv has 539241 non-null elements.
Input file daily_20241108T0937_009.csv has 586794 non-null elements.
Input file daily_20241108T0938_010.csv has 547088 non-null elements.
Input file daily_20241108T0938_011.csv has 492207 non-null elements.
615620


In [46]:
# read our metadata file
meta_df = pd.read_csv("bc_station_metadata/metadata_20241115T0843.csv", dtype={'Station Number': 'str'}, low_memory=False)
# change header "Station Number" -> "Station ID"
meta_df.rename(columns={"Station Number":"Station ID"}, inplace=True)

In [47]:
meta_df['Station ID']

0       07EA001
1       07EA002
2       07EA004
3       07EA005
4       07EA006
         ...   
2468    10CD003
2469    10CD004
2470    10CD005
2471    10CD006
2472    10DA001
Name: Station ID, Length: 2473, dtype: object

In [50]:
combined_data = pd.read_csv("combined_data.csv")
combined_data

,Unnamed: 0,Station ID,Date,Daily Discharge,SYM1,Daily Water Level,SYM2,Station Name,Province,Status,...,Regulation,Data Type,Operation Schedule,Sediment,RHBN,Real-Time,Datum Name,Publishing Office,Operating Agency,Contributed
0,0,07EA005,2011/01/01,60.20,B,1.075,NaN,FINLAY RIVER ABOVE AKIE RIVER,BC,Active,...,N,Flow and Level,Continuous,N,N,Y,ASSUMED DATUM,VANCOUVER,NaN,N
1,1,07EA005,2011/01/02,59.70,B,1.082,NaN,FINLAY RIVER ABOVE AKIE RIVER,BC,Active,...,N,Flow and Level,Continuous,N,N,Y,ASSUMED DATUM,VANCOUVER,NaN,N
2,2,07EA005,2011/01/03,59.20,B,1.098,NaN,FINLAY RIVER ABOVE AKIE RIVER,BC,Active,...,N,Flow and Level,Continuous,N,N,Y,ASSUMED DATUM,VANCOUVER,NaN,N
3,3,07EA005,2011/01/04,58.60,B,1.110,NaN,FINLAY RIVER ABOVE AKIE RIVER,BC,Active,...,N,Flow and Level,Continuous,N,N,Y,ASSUMED DATUM,VANCOUVER,NaN,N
4,4,07EA005,2011/01/05,58.10,B,1.109,NaN,FINLAY RIVER ABOVE AKIE RIVER,BC,Active,...,N,Flow and Level,Continuous,N,N,Y,ASSUMED DATUM,VANCOUVER,NaN,N
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1048009,1048009,10DA001,2023/12/27,1.66,B,5.314,NaN,PETITOT RIVER BELOW HIGHWAY NO. 77,BC,Active,...,N,Flow and Level,Continuous,N,N,Y,ASSUMED DATUM,YELLOWKNIFE,NaN,N
1048010,1048010,10DA001,2023/12/28,1.64,B,5.324,NaN,PETITOT RIVER BELOW HIGHWAY NO. 77,BC,Active,...,N,Flow and Level,Continuous,N,N,Y,ASSUMED DATUM,YELLOWKNIFE,NaN,N
1048011,1048011,10DA001,2023/12/29,1.61,B,5.388,NaN,PETITOT RIVER BELOW HIGHWAY NO. 77,BC,Active,...,N,Flow and Level,Continuous,N,N,Y,ASSUMED DATUM,YELLOWKNIFE,NaN,N
1048012,1048012,10DA001,2023/12/30,1.59,B,5.456,NaN,PETITOT RIVER BELOW HIGHWAY NO. 77,BC,Active,...,N,Flow and Level,Continuous,N,N,Y,ASSUMED DATUM,YELLOWKNIFE,NaN,N


In [142]:
combined_data = pd.read_csv("combined_data.csv")
combined_data['Date'] = pd.to_datetime(combined_data['Date'], format="%Y/%m/%d")#.dt.date

# do some manual filtering of stations with data over too small a period of time
combined_data = combined_data[combined_data['Station ID'] != '08GA026']
combined_data = combined_data[combined_data['Station ID'] != '08HE001']
combined_data = combined_data[combined_data['Station ID'] != '08LG070']
combined_data = combined_data[combined_data['Station ID'] != '08LF023']
combined_data = combined_data[combined_data['Station ID'] != '08KE018']
combined_data = combined_data[combined_data['Station ID'] != '08ND021']
combined_data = combined_data[combined_data['Station ID'] != '08MF035']
combined_data = combined_data[combined_data['Station ID'] != '08CE005']
combined_data = combined_data[combined_data['Station ID'] != '07ED001']
combined_data = combined_data[combined_data['Station ID'] != '08GD010']
combined_data = combined_data[combined_data['Station ID'] != '08MG028']
combined_data = combined_data[combined_data['Station ID'] != '08NK030']
combined_data = combined_data[combined_data['Station ID'] != '09AA006']
combined_data = combined_data[combined_data['Station ID'] != '08LF033']
combined_data = combined_data[combined_data['Station ID'] != '08HD035']
combined_data = combined_data[combined_data['Station ID'] != '08FA002']
combined_data = combined_data[combined_data['Station ID'] != '08MD013']
combined_data = combined_data[combined_data['Station ID'] != '08KH001']
combined_data = combined_data[combined_data['Station ID'] != '08PA012']
combined_data = combined_data[combined_data['Station ID'] != '08LD001']
combined_data = combined_data[combined_data['Station ID'] != '08EG012']
combined_data = combined_data[combined_data['Station ID'] != '08EG019']
combined_data = combined_data[combined_data['Station ID'] != '07FC003']
combined_data = combined_data[combined_data['Station ID'] != '08NP003']
combined_data = combined_data[combined_data['Station ID'] != '07FD019']
combined_data = combined_data[combined_data['Station ID'] != '08LF094']
combined_data = combined_data[combined_data['Station ID'] != '10CD004']
combined_data = combined_data[combined_data['Station ID'] != '08NM146']
combined_data = combined_data[combined_data['Station ID'] != '08KE024']
combined_data = combined_data[combined_data['Station ID'] != '08LG056']
combined_data = combined_data[combined_data['Station ID'] != '08KH019']
combined_data = combined_data[combined_data['Station ID'] != '08KA009']
combined_data = combined_data[combined_data['Station ID'] != '08KH010']
combined_data = combined_data[combined_data['Station ID'] != '10CD005']
combined_data = combined_data[combined_data['Station ID'] != '07FC001']
combined_data = combined_data[combined_data['Station ID'] != '08MF005']



minDate = combined_data['Date'].min()
maxDate = combined_data['Date'].max()
minStation = ""
maxStation = ""
for station in combined_data['Station ID'].unique():
    stationMin = combined_data[combined_data["Station ID"] == station]['Date'].min()
    stationMax = combined_data[combined_data["Station ID"] == station]['Date'].max()
    if stationMin > minDate:
        minDate = stationMin
        minStation = station
    if stationMax < maxDate:
        maxDate = stationMax
        maxStation = station

print(f'MinDate: {minDate}\nMaxDate: {maxDate}')
print(minStation, maxStation)


MinDate: 2013-01-01 00:00:00
MaxDate: 2023-12-31 00:00:00
10DA001 


In [174]:
# TODO before pivot, get rid of some more useless columns

columns = ['Date','Station ID','Station Name','Province','Latitude','Longitude','Daily Discharge',
           'SYM1','Daily Water Level','SYM2']
combined_data = combined_data[columns]

In [175]:
pivot_df = combined_data.pivot(columns="Station ID", index="Date")

In [177]:
x = pd.date_range(start="1/1/2013", end="12/31/2023")

In [178]:
pivot_df = pivot_df.reindex(x)

In [202]:
pivot_df['Daily Water Level']

Station ID,07EA005,07EB002,07EC002,07EC003,07ED003,07FA004,07FA005,07FA006,07FB001,07FB002,...,08NN028,09AA013,09AE003,10BE007,10BE009,10CA001,10CB001,10CD001,10CD003,10DA001
2013-01-01,1.113,1.794,1.801,2.354,1.167,1.914,0.001,0.874,2.451,8.022,...,0.841,10.760,1.039,2.752,7.644,6.890,1.629,-0.029,0.202,5.773
2013-01-02,1.105,1.777,1.806,2.347,1.139,2.055,0.040,0.868,2.446,8.015,...,0.847,10.761,1.023,2.743,7.640,6.890,1.627,-0.035,0.196,5.781
2013-01-03,1.091,1.757,1.806,2.335,1.128,2.232,0.077,0.866,2.433,8.006,...,NaN,10.758,1.016,2.768,7.637,6.880,1.621,-0.037,0.180,5.782
2013-01-04,1.081,1.741,1.801,2.320,1.115,2.254,0.081,0.868,2.412,7.994,...,NaN,10.758,1.000,2.726,7.636,6.880,1.619,-0.042,0.177,5.783
2013-01-05,1.073,1.725,1.798,2.310,1.118,2.279,0.064,0.862,2.394,7.979,...,NaN,10.754,0.988,2.708,7.632,6.880,1.616,-0.044,0.176,5.785
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2023-12-27,0.244,1.922,1.733,2.498,0.627,0.146,-0.212,0.294,2.574,7.906,...,0.792,10.888,0.914,2.091,7.549,6.518,1.520,-0.123,0.227,5.314
2023-12-28,0.253,1.913,1.731,2.484,0.605,0.144,-0.136,0.287,2.553,8.052,...,0.788,10.883,0.997,2.091,7.550,6.535,1.513,-0.139,0.224,5.324
2023-12-29,0.256,1.894,1.727,2.452,0.684,0.144,0.002,0.281,2.389,8.125,...,0.787,10.884,0.926,2.130,7.534,6.537,1.496,-0.081,0.221,5.388
2023-12-30,0.273,1.900,1.724,2.400,0.708,0.140,-0.062,0.268,2.349,8.132,...,0.787,10.887,0.910,2.151,7.556,6.530,1.525,-0.085,0.216,5.456


In [ ]:
pivot_df['Daily Water Level'].dropna(axis=0, thresh=169)
# so if we allow some dates to miss data for up to 31 stations, then we can use this date range

Station ID,07EA005,07EB002,07EC002,07EC003,07ED003,07FA004,07FA005,07FA006,07FB001,07FB002,...,08NN028,09AA013,09AE003,10BE007,10BE009,10CA001,10CB001,10CD001,10CD003,10DA001
2013-01-01,1.113,1.794,1.801,2.354,1.167,1.914,0.001,0.874,2.451,8.022,...,0.841,10.760,1.039,2.752,7.644,6.890,1.629,-0.029,0.202,5.773
2013-01-02,1.105,1.777,1.806,2.347,1.139,2.055,0.040,0.868,2.446,8.015,...,0.847,10.761,1.023,2.743,7.640,6.890,1.627,-0.035,0.196,5.781
2013-01-03,1.091,1.757,1.806,2.335,1.128,2.232,0.077,0.866,2.433,8.006,...,NaN,10.758,1.016,2.768,7.637,6.880,1.621,-0.037,0.180,5.782
2013-01-04,1.081,1.741,1.801,2.320,1.115,2.254,0.081,0.868,2.412,7.994,...,NaN,10.758,1.000,2.726,7.636,6.880,1.619,-0.042,0.177,5.783
2013-01-05,1.073,1.725,1.798,2.310,1.118,2.279,0.064,0.862,2.394,7.979,...,NaN,10.754,0.988,2.708,7.632,6.880,1.616,-0.044,0.176,5.785
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2023-12-27,0.244,1.922,1.733,2.498,0.627,0.146,-0.212,0.294,2.574,7.906,...,0.792,10.888,0.914,2.091,7.549,6.518,1.520,-0.123,0.227,5.314
2023-12-28,0.253,1.913,1.731,2.484,0.605,0.144,-0.136,0.287,2.553,8.052,...,0.788,10.883,0.997,2.091,7.550,6.535,1.513,-0.139,0.224,5.324
2023-12-29,0.256,1.894,1.727,2.452,0.684,0.144,0.002,0.281,2.389,8.125,...,0.787,10.884,0.926,2.130,7.534,6.537,1.496,-0.081,0.221,5.388
2023-12-30,0.273,1.900,1.724,2.400,0.708,0.140,-0.062,0.268,2.349,8.132,...,0.787,10.887,0.910,2.151,7.556,6.530,1.525,-0.085,0.216,5.456
